In [2]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from connection_alchemy import connect_to_db

CSV_PATH = "/Users/adrienblanquer/Downloads/Contacts-Export-30_04_2026-09_24_43.csv"
TABLE = "easybill.contacts"

df = pd.read_csv(CSV_PATH, sep=";", dtype=str, keep_default_na=False)
df = df[df["Typ"] == "Kunde"].copy()
df = df.drop(columns=["Leere Spalte"])
df.columns = [f"Kontakt: {c}" for c in df.columns]

INT_COLS = [
    "Kontakt: Kontakt ID",
    "Kontakt: Persönlich/Vertraulich",
    "Kontakt: Persönlich/Vertraulich-Lieferanschrift",
    "Kontakt: Skonto (Tage)",
    "Kontakt: Zahlungsziel (Tage)",
    "Kontakt: Lieferantennr. beim Kontakt",
    "Kontakt: Archiviert",
]
for c in INT_COLS:
    df[c] = pd.to_numeric(df[c].replace("", pd.NA), errors="coerce").astype("Int64")

for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].where(df[c] != "", None)

assert df["Kontakt: Kundennummer"].is_unique, "Kundennummer must be unique among Kunde rows"
print(f"CSV rows ready to upsert: {len(df)}")
df.head(2)

CSV rows ready to upsert: 3504


,Kontakt: Kontakt ID,Kontakt: Kundennummer,Kontakt: Lieferantennummer,Kontakt: Abteilung,Kontakt: Anrede,Kontakt: Persönlich/Vertraulich,Kontakt: Titel,Kontakt: Vorname,Kontakt: Name,Kontakt: Geburtsdatum,...,Kontakt: Rabatt (%/Betrag),Kontakt: Zahlungsziel (Tage),Kontakt: Umsatzsteueroption,Kontakt: Umsatzsteueroption ID,Kontakt: Typ,Kontakt: Ansprechpartner ID,Kontakt: Dokumentformat beim Versenden,Kontakt: Referenz des Käufers,Kontakt: Lieferantennr. beim Kontakt,Kontakt: Archiviert
0,1094312433,10081,,,,0,,,,,...,,<NA>,Umsatzsteuerpflichtig,NULL,Kunde,,default,,<NA>,1
1,726404676,100000002,,,Frau,0,,Tara,Moshtael,,...,,<NA>,Umsatzsteuerpflichtig,NULL,Kunde,,default,,<NA>,1


In [3]:
from sqlalchemy import text

conn = connect_to_db()

db_cols = [r[0] for r in conn.execute(text("""
    select column_name
    from information_schema.columns
    where table_schema = 'easybill' and table_name = 'contacts'
    order by ordinal_position
""")).fetchall()]

csv_only = [c for c in df.columns if c not in db_cols]
db_only = [c for c in db_cols if c not in df.columns]
print("CSV columns missing from DB:", csv_only)
print("DB columns missing from CSV (will stay NULL on inserts):", db_only)
assert not csv_only, "CSV has columns the DB does not — refusing to proceed"

CSV columns missing from DB: []
DB columns missing from CSV (will stay NULL on inserts): []


In [5]:
key = "Kontakt: Kundennummer"
csv_kundennr = df[key].tolist()

existing_total = conn.execute(text(f"select count(*) from {TABLE}")).scalar()
existing = pd.read_sql(
    text(f'select * from {TABLE} where "Kontakt: Kundennummer" = ANY(:knrs)'),
    conn,
    params={"knrs": csv_kundennr},
)
will_update = len(existing)
will_insert = len(df) - will_update

print(f"Existing rows in {TABLE}: {existing_total}")
print(f"CSV rows to upsert:       {len(df)}")
print(f"  → matched (overwrite):  {will_update}")
print(f"  → unmatched (insert):   {will_insert}")
print(f"  → untouched in DB:      {existing_total - will_update}")
print()
print("Match key: \"Kontakt: Kundennummer\" only.")
print("Overwrite = the existing row is DELETED and re-INSERTED from CSV. Every column")
print("is replaced with the CSV value; nothing is merged column-by-column.")
print()
print(f"DB-only columns (no CSV source → reset to NULL on overwritten rows): {db_only}")
if db_only:
    losing = {c: int(existing[c].notna().sum()) for c in db_only if c in existing.columns}
    print(f"  overwritten rows currently holding values in those columns: {losing}")
print()

# Per-column diff: among the matched rows, how many will see a value change
def _norm(s):
    return s.where(s.notna() & (s.astype(str) != ""), None)

merged = existing.merge(df, on=key, suffixes=("_old", "_new"))
diff_cols = []
for c in df.columns:
    if c == key:
        continue
    n = int((_norm(merged[f"{c}_old"]).astype(object) != _norm(merged[f"{c}_new"]).astype(object)).sum())
    if n:
        diff_cols.append((c, n))
diff_cols.sort(key=lambda x: -x[1])

print(f"Of the {will_update} matched rows, columns that will actually change: {len(diff_cols)}")
print("Top 15 by # rows changed:")
for c, n in diff_cols[:15]:
    print(f"  {n:5d}  {c}")

Existing rows in easybill.contacts: 3212
CSV rows to upsert:       3504
  → matched (overwrite):  3195
  → unmatched (insert):   309
  → untouched in DB:      17

Match key: "Kontakt: Kundennummer" only.
Overwrite = the existing row is DELETED and re-INSERTED from CSV. Every column
is replaced with the CSV value; nothing is merged column-by-column.

DB-only columns (no CSV source → reset to NULL on overwritten rows): []

Of the 3195 matched rows, columns that will actually change: 68
Top 15 by # rows changed:
   3195  Kontakt: Lieferantennummer
   3195  Kontakt: Abteilung
   3195  Kontakt: Geburtsdatum
   3195  Kontakt: Registergericht
   3195  Kontakt: Registernummer
   3195  Kontakt: Kontoinhaber
   3195  Kontakt: Bankname
   3195  Kontakt: Kontonummer
   3195  Kontakt: Bankleitzahl
   3195  Kontakt: Zusatz 2-Lieferanschrift
   3195  Kontakt: Preisgruppe
   3195  Kontakt: SEPA-Lastschriftverfahren
   3195  Kontakt: Skonto (%)
   3195  Kontakt: Rabatt (%/Betrag)
   3195  Kontakt: Ansp

In [12]:
conn.rollback()

In [ ]:
try:
    deleted = conn.execute(
        text(f'delete from {TABLE} where "Kontakt: Kundennummer" = ANY(:knrs)'),
        {"knrs": csv_kundennr},
    ).rowcount
    df.to_sql(
        "contacts",
        conn,
        schema="easybill",
        if_exists="append",
        index=False,
        method="multi",
        chunksize=500,
    )
    after = conn.execute(text(f"select count(*) from {TABLE}")).scalar()
    conn.commit()
    print(f"deleted {deleted} rows, inserted {len(df)} rows")
    print(f"final row count: {after}")
except Exception:
    conn.rollback()
    raise

In [ ]:
try:
    conn.execute(text(
        "create table bas_firms.easybill_medisoft_1 as "
        "select * from bas_firms.easybill_medisoft"
    ))
    n = conn.execute(text("select count(*) from bas_firms.easybill_medisoft_1")).scalar()
    conn.commit()
    print(f"backed up {n} rows to bas_firms.easybill_medisoft_1")
except Exception:
    conn.rollback()
    raise

In [18]:
EM_TABLE = "bas_firms.easybill_medisoft"

existing_em_ids = {
    r[0]
    for r in conn.execute(
        text(f"select distinct easybill_id from {EM_TABLE}")
    ).fetchall()
}

missing = [k for k in df[key].tolist() if k not in existing_em_ids]
print(f"Kundennummern in CSV not yet in {EM_TABLE}: {len(missing)}")
print("First 10:", missing[:10])

try:
    inserted = conn.execute(
        text(f"insert into {EM_TABLE} (id, easybill_id, medisoft_id) values (:k, :k, NULL)"),
        [{"k": k} for k in missing],
    ).rowcount
    after = conn.execute(text(f"select count(*) from {EM_TABLE}")).scalar()
    conn.commit()
    print(f"inserted {inserted} placeholder rows (id=easybill_id=Kundennummer, medisoft_id=NULL)")
    print(f"final row count in {EM_TABLE}: {after}")
except Exception:
    conn.rollback()
    raise

Kundennummern in CSV not yet in bas_firms.easybill_medisoft: 299
First 10: ['100020030', '101000001', '101000025', '104000041', '106020037', '107030007', '110020016', '111000017', '111001008', '111001012']
inserted 299 placeholder rows (id=easybill_id=Kundennummer, medisoft_id=NULL)
final row count in bas_firms.easybill_medisoft: 4250
